In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (ToxinPred 2.0)

This notebook processes and standardizes the **ToxinPred 2.0** dataset into a clean and unified format suitable for downstream analysis and machine learning. The raw data are provided as multiple FASTA files, where peptide toxicity labels are **inferred from the source filenames**.

- **Toxic effect / endpoint:** toxic
- **Source:** ToxinPred 2.0
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Reads all FASTA files** associated with the ToxinPred 2.0 dataset.
- **Infers binary toxicity labels from filenames**:
  - files containing `Negative` (case-insensitive) are labeled as `0` (non-toxic),
  - all other files are labeled as `1` (toxic).
- **Standardizes the dataset schema** to two columns: `sequence` and `label`.
- **Performs duplicate sequence quality control**:
  - retains a single representative when duplicate sequences share the same label,
  - flags sequences with conflicting labels as erroneous.
- **Generates dataset-level metadata** using `raw_data_description.xlsx`.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv` containing the deduplicated dataset,
  - `metadata.json` summarizing dataset provenance and statistics.

In [2]:
name_source = "ToxinPred 2.0"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
dfs = []
for file in (Path(PATH_INPUT) / name_source).glob("*"):
    df = read_fasta_doc(file)
    df["source_file"] = file.name
    dfs.append(df)
df_toxinpred = pd.concat(dfs, ignore_index=True)

In [4]:
df_toxinpred = (
    df_toxinpred
    .assign(
        label=lambda d: d["source_file"]
            .str.contains("Negative", case=False, na=False)
            .map({True: 0, False: 1})
    )
    [["sequence","label"]]
)
df_toxinpred.shape

(41478, 2)

- Checking duplicates

In [5]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_toxinpred, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [6]:
df_full.shape

(35242, 2)

In [7]:
df_errors.shape

(0, 1)

- Working with metada

In [8]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [9]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_toxinpred)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'GNU general public license',
 'year of publication': 2022,
 'last update date': datetime.datetime(2022, 5, 21, 0, 0),
 'download date': Timestamp('2024-08-01 00:00:00'),
 'file format': 'fasta',
 'peptide property': 'toxic',
 'dataset information': 'Negative;Positive',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from Swiss-Prot;No information',
 'repository or server': 'https://webs.iiitd.edu.in/raghava/toxinpred2/',
 'publication': 'https://academic.oup.com/bib/article/23/5/bbac174/6590152?login=false',
 'number_of_raw_sequences': 41478,
 'number_of_sequences_retained': 35242,
 'number_of_positive_sequences': 8233,
 'number_of_negative_sequences': 27009,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [10]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [11]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)